# Evaluating generated elaborations with automatic metrics

This notebook focuses on the **automatic evaluation of generated elaborations** using multiple metrics:

- **BLEU** metric, which has been widely used in previous studies to evaluate elaborations
- **BERTScore**, which measures semantic similarity between the generated elaboration and the reference.
- **BARTScore**, which captures contextual relevance and overall coherence

Users can follow the evaluation process step by step in this notebook to compare different models and assess their performance.
Alternatively, if you prefer to execute the full pipeline automatically, you can run the script via terminal, specifying the type of score to be calculated:  

```bash
python calculate_scores.py --score bleu
python calculate_scores.py --score bert
python calculate_scores.py --score bart

# Load the data

In [8]:
models = ["llama-ft","bart-ft","llama-instruct-few-shot"]

setting_ds_dict = {
    "base": ["c2s","c2sp","c4s","c4sp"],
    "masked": ["c2s","c2sp","c4s","c4sp"],
    "subject":["c2s","c2sp","c4s","c4sp"],
    "target-phrase":["c2s","c2sp","c4s","c4sp],
    "target-sent":["c2s","c2sp","c4s","c4sp"],
    "target-sent-target":["c2s","c2sp","c4s","c4sp"],
    "target-sent-subject":["c2s","c2sp","c4s","c4sp"],
}

# for results inspection
results_setting_ds_dict = {
    "base": ["cs","c4s"],
    "subject":["c4s"],
    "target-phrase":["c4s"],
    "target-sent":["c4s"],
    "target-sent-target":["c4sp"],
    "target-sent-subject":["c2sp","c4spo"],
}

# additional calculation
setting_ds_dict = {
    "base": ["c2s","c2sp","c4s","c4sp","c2o","c2op"],
    "masked": ["c2s","c2sp","c4s","c4sp"],
    "target-phrase":["c2s","c2sp","c4s","c4sp","c2o","c2op"],
    "target-sent":["c2s","c2sp","c4s","c4sp","c2o","c2op"],
    "target-sent-target":["c2s","c2sp","c4s","c4sp","c2o","c2op"]
}

# Load results df

In [3]:
import pandas as pd
bart_ft_res = pd.read_csv("../data/results/bart-ft-results.csv")
llama_ft_res = pd.read_csv("../data/results/llama-ft-results.csv")
llama_instr_res = pd.read_csv("../data/results/llama-instruct-few-shot-results.csv")
llama_instr_prompt_res = pd.read_csv("../data/results/llama-instruct-few-shot-prompt-results.csv")

# Modify results dataframe

## Initialize columns in the results dfs (BERTScore)

In [ ]:
df_res = pd.read_csv(f"../data/results/{model}-results.csv")
settings = list(dict.fromkeys(["-".join(col.split("-")[:-1]) for col in df_res.columns if "-" in col]))
cols_to_add = [f"{col_name}-bs-prec" for col_name in settings] + \
              [f"{col_name}-bs-rec" for col_name in settings] + \
              [f"{col_name}-bs-f1" for col_name in settings]
for col in cols_to_add:
    df_res[col] = None 

## Rename columns

In [34]:
# rename certain columns
df_res.columns = [
    col.replace('-bs-rec', '-bsrec').replace('-bs-prec', '-bsprec').replace('-bs-f1', '-bsf1')
    if col.endswith(('-bs-rec', '-bs-prec', '-bs-f1')) else col
    for col in df_res.columns
]

print(df_res.columns)

Index(['dataset', 'base-b1', 'base-b2', 'masked-b1', 'masked-b2', 'subject-b1',
       'subject-b2', 'target-phrase-b1', 'target-phrase-b2', 'target-sent-b1',
       'target-sent-b2', 'target-sent-target-b1', 'target-sent-target-b2',
       'target-sent-subject-b1', 'target-sent-subject-b2', 'base-bsprec',
       'masked-bsprec', 'subject-bsprec', 'target-phrase-bsprec',
       'target-sent-bsprec', 'target-sent-target-bsprec',
       'target-sent-subject-bsprec', 'base-bsrec', 'masked-bsrec',
       'subject-bsrec', 'target-phrase-bsrec', 'target-sent-bsrec',
       'target-sent-target-bsrec', 'target-sent-subject-bsrec', 'base-bsf1',
       'masked-bsf1', 'subject-bsf1', 'target-phrase-bsf1', 'target-sent-bsf1',
       'target-sent-target-bsf1', 'target-sent-subject-bsf1'],
      dtype='object')


# Create scores dataframe

In [4]:
def create_scores_df(df_gen):
    df_scores = pd.DataFrame({
        'source_text': df_gen['source_text'] if 'source_text' in df_gen else None,
        'target_sentence': (
            df_gen['target_sentence_4o'] if 'target_sentence_4o' in df_gen
            else df_gen['target_sentence'] if 'target_sentence' in df_gen
            else None
        ),
        'target_sentence_target': df_gen['target_sentence_target'] if 'target_sentence_target' in df_gen else None,
        'subject': df_gen['subject'] if 'subject' in df_gen else None,
        'target-phrase': df_gen['target-phrase'] if 'target-phrase' in df_gen else None,
        'elaboration_sentence': df_gen['elaboration_sentence'],
        'pred_elaboration': df_gen['pred_elaboration'],
    })
    return df_scores

# BLEU-4 (EASSE package)

In [25]:
from tqdm.notebook import tqdm
from easse.bleu import corpus_bleu
import numpy as np

bleu_scores_easse = []

for index, row in tqdm(df_gen.iterrows(), total=len(df_gen)):
    s_content = row['elaboration_sentence'] 
    prediction = row['pred_elaboration'] # "prediction" for BART
    
    bleu_score_easse = corpus_bleu(
        sys_sents=[prediction],
        refs_sents=[[s_content]]
    )
    
    bleu_scores_easse.append(bleu_score_easse)

print(f"Average BLEU score: {np.mean(bleu_scores_easse):.3f}")

  0%|          | 0/116 [00:00<?, ?it/s]

Average BLEU score: 4.965


# BLEU-1 & BLEU-2 (nltk + tokenizer-13A)

## Corpus bleu

In [4]:
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
from sacrebleu.tokenizers.tokenizer_13a import Tokenizer13a
from dataset_utils import create_scores_df
from tqdm import tqdm

# 13a tokenizer
tokenizer = Tokenizer13a()
smoothing_function = SmoothingFunction().method1

for model in models: 
    df_res = pd.read_csv(f"../data/results/{model}-results.csv")
    for setting_key, ds_values in setting_ds_dict.items():
        for ds in ds_values:
            
            all_refs = []
            all_preds = []
            output_name = f"{ds}-{setting_key}-{num_examples}"
            df_gen = pd.read_csv(f"../data/gen_predictions/predictions_{model}-{output_name}.csv")
 
            for index, row in tqdm(df_gen.iterrows(), total=len(df_gen)):
                ref = row['elaboration_sentence']
                prediction = row['pred_elaboration'] # "prediction" for BART
            
                # Tokenize
                tokenized_ref = tokenizer(ref).split()
                tokenized_pred = tokenizer(prediction).split()
                
                all_refs.append([tokenized_ref]) 
                all_preds.append(tokenized_pred)
            
            bleu1_score = corpus_bleu(all_refs, all_preds, weights=(1.0, 0, 0, 0), smoothing_function=smoothing_function)  # 1-gram
            bleu2_score = corpus_bleu(all_refs, all_preds, weights=(0.5, 0.5, 0, 0), smoothing_function=smoothing_function)  # 2-gram
            bleu4_score = corpus_bleu(all_refs, all_preds, weights=(0.25, 0.25, 0.25, 0.25), smoothing_function=smoothing_function)  # 4-gram

            idx = df_res.index[df_res["dataset"] == ds].tolist()[0]
            df_res.at[idx, f"{setting_key}-{num_examples}-b1"] = round(bleu1_score*100,3)
            df_res.at[idx, f"{setting_key}-{num_examples}-b2"] = round(bleu2_score*100,3)
            print(f"{model}-{output_name}: {round(bleu1_score*100,3)}")
            print(f"{model}-{output_name}: {round(bleu2_score*100,3)}")
    
    df_res.to_csv(f"../data/results/{model}-results.csv",index=False)
    print(f"Results saved for {model}")

100%|██████████████████████████████████████| 116/116 [00:00<00:00, 10129.48it/s]


llama-instruct-few-shot-c2s-base-n6: 16.284
llama-instruct-few-shot-c2s-base-n6: 5.439


100%|██████████████████████████████████████| 116/116 [00:00<00:00, 16714.39it/s]


llama-instruct-few-shot-c2sp-base-n6: 16.451
llama-instruct-few-shot-c2sp-base-n6: 5.631


100%|███████████████████████████████████████| 116/116 [00:00<00:00, 9579.05it/s]


llama-instruct-few-shot-c4s-base-n6: 15.158
llama-instruct-few-shot-c4s-base-n6: 3.637


100%|██████████████████████████████████████| 116/116 [00:00<00:00, 16469.97it/s]

llama-instruct-few-shot-c4sp-base-n6: 14.426
llama-instruct-few-shot-c4sp-base-n6: 3.505
Results saved for llama-instruct-few-shot


## Sentence bleu

In [ ]:
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from sacrebleu.tokenizers.tokenizer_13a import Tokenizer13a
from transformers import BartTokenizer
from tqdm.notebook import tqdm

# 13a tokenizer
tokenizer = Tokenizer13a()

smoothing_function = SmoothingFunction().method1

setting_ds_dict = {
    "base": ["c2sp","c4s","c4sp"],
    "masked": ["c2sp","c4sp","c4s"],
    "target-phrase":["c2sp","c4sp","c4s"],
    "target-sent":["c2s","c2sp","c4s","c4sp"],
    "target-sent-target":["c2sp","c4sp","c4s"]
}

for model in models: 
    for setting_key, ds_values in setting_ds_dict.items():
        for ds in ds_values:

            bleu_scores_1 = []
            bleu_scores_2 = []
            
            output_name = f"{ds}-{setting_key}"
            df_gen = pd.read_csv(f"../data/gen_predictions/predictions_{model}-{output_name}.csv")
            df_scores = create_scores_df(df_gen)
            for index, row in tqdm(df_gen.iterrows(), total=len(df_gen)):
                ref = row['elaboration_sentence']
                prediction = row['pred_elaboration']
            
                # tokenize
                tokenized_ref = tokenizer(ref).split()
                tokenized_pred = tokenizer(prediction).split()
                    
                bleu_score_1 = sentence_bleu([tokenized_ref],tokenized_pred,weights=(1, 0, 0, 0),smoothing_function=smoothing_function) # 1-gram
                bleu_score_2 = sentence_bleu([tokenized_ref],tokenized_pred,weights=(0.5, 0.5, 0, 0), smoothing_function=smoothing_function) # 2-gram
                bleu_scores_1.append(round(bleu_score_1,3))
                bleu_scores_2.append(round(bleu_score_2,3))

            df_scores["b1"] = bleu_scores_1
            df_scores["b2"] = bleu_scores_2
            df_scores.to_csv(f"../data/bleu_scores/bleu_scores_{model}-{output_name}.csv",index=False)
            print(f"Results saved for {model}")

# BERTScore

In [5]:
from tqdm.notebook import tqdm
from bert_score import BERTScorer
import numpy as np
import pandas as pd
from transformers import logging

scorer = BERTScorer(model_type='bert-base-uncased',device='cuda:0')
setting_ds_dict = {"base":["c2sp"]}
models = ["llama-instruct-few-shot"]
num_examples = "n6"

for model in models: 
    df_res = pd.read_csv(f"../data/results/{model}-results.csv")
    for setting_key, ds_values in setting_ds_dict.items():
        for ds in ds_values:
            
            bert_scores_f1 = []
            
            output_name = f"{ds}-{setting_key}-{num_examples}"
            df_gen = pd.read_csv(f"../data/gen_predictions/predictions_{model}-{output_name}.csv")
            df_scores = create_scores_df(df_gen)
            
            for index, row in tqdm(df_gen.iterrows(), total=len(df_gen)):
                elaboration = row['elaboration_sentence']
                prediction = row['pred_elaboration']
                
                #  BERTScore for this pair
                P, R, F1 = scorer.score(
                    cands=[prediction],  
                    refs=[elaboration],              
                )
                
                bert_scores_f1.append(F1.mean().item())

            # save scores for each pair
            df_scores["bsf1"] = bert_scores_f1
            df_scores.to_csv(f"../data/bert_scores/bert_scores_{model}-{output_name}.csv",index=False)
            print(f"Results saved for {model}")
            
            # save average scores to models general results 
            avg_precision = np.mean(bert_scores_precision)
            avg_recall = np.mean(bert_scores_recall)
            avg_f1 = np.mean(bert_scores_f1)

            idx = df_res.index[df_res["dataset"] == ds].tolist()[0]
            df_res.at[idx, f"{setting_key}-{num_examples}-bsprec"] = round(avg_precision,3)
            df_res.at[idx, f"{setting_key}-{num_examples}-bsrec"] = round(avg_recall,3)
            df_res.at[idx, f"{setting_key}-bsf1"] = round(avg_f1,3)
            print(f"{model}-{ds}-{setting_key}: {round(avg_f1,3)}")

    df_res.to_csv(f"../data/results/{model}-results.csv",index=False)
    print(f"Results saved for {model}")

  0%|          | 0/116 [00:00<?, ?it/s]

Results saved for llama-instruct-few-shot


# BARTScore

https://github.com/neulab/BARTScore

In [ ]:
import sys
import os
from model_utils import BARTScorer
from tqdm.notebook import tqdm
import numpy as np

bart_scorer = BARTScorer(device='cuda:0')

models = ["llama-ft","bart-ft"]
#num_examples = "n6"

for model in models: 
    df_res = pd.read_csv(f"data/results/{model}-results.csv")
    for setting_key, ds_values in setting_ds_dict.items():
        for ds in ds_values:
            bart_scores = []
            output_name = f"{ds}-{setting_key}"
            df_gen = pd.read_csv(f"../data/gen_predictions/predictions_{model}-{output_name}.csv")
            df_scores = create_scores_df(df_gen)
        
            for index, row in tqdm(df_gen.iterrows(), total=len(df_gen)):
                reference = row['elaboration_sentence']  # reference text (r)
                hypothesis = row['pred_elaboration']    # generated text (h)
                
                # precision (r → h)
                precision_score = bart_scorer.score(
                    srcs=[reference],  # r as source
                    tgts=[hypothesis], # h as target
                    batch_size=1
                )[0]
                
                # recall (h → r)
                recall_score = bart_scorer.score(
                    srcs=[hypothesis],  # h as source
                    tgts=[reference],   # r as target
                    batch_size=1
                )[0]
                
                # f1 score as the average of precision and recall
                f1_score = (precision_score + recall_score) / 2
                bart_scores.append(f1_score)
            
            # save result for each pair
            df_scores["bartscore"] = bart_scores
            df_scores.to_csv(f"../data/bart_scores/bart_scores_{model}-{output_name}.csv",index=False)
            print(f"Results saved for {model}-{output_name}")
        
            # average score
            avg_score = np.mean(bart_scores)
            idx = df_res.index[df_res["dataset"] == ds].tolist()[0]
            df_res.at[idx, f"{setting_key}-{num_examples}-bartscore"] = round(avg_score,3)
            print(f"{model}-{ds}-{setting_key}-{num_examples}: {round(avg_score,3)}")
    
    df_res.to_csv(f"data/results/{model}-results.csv",index=False)
    print(f"Results saved for {model}")

# Prompt evaluation

In [ ]:
def create_scores_df(df_gen):
    df_scores = pd.DataFrame({
        'source_text': df_gen['source_text'] if 'source_text' in df_gen else None,
        'target_sentence': (
            df_gen['target_sentence_4o'] if 'target_sentence_4o' in df_gen
            else df_gen['target_sentence'] if 'target_sentence' in df_gen
            else None
        ),
        'target_sentence_target': df_gen['target_sentence_target'] if 'target_sentence_target' in df_gen else None,
        'subject': df_gen['subject'] if 'subject' in df_gen else None,
        'target-phrase': df_gen['target-phrase'] if 'target-phrase' in df_gen else None,
        'elaboration_sentence': df_gen['elaboration_sentence'],
        'pred_elaboration': df_gen['pred_elaboration'],
    })
    return df_scores

## BLEU score

In [4]:
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
from sacrebleu.tokenizers.tokenizer_13a import Tokenizer13a
import pandas as pd
from tqdm import tqdm

# 13a tokenizer
tokenizer = Tokenizer13a()
smoothing_function = SmoothingFunction().method1

# prompt evaluation
models = ["llama-instruct-few-shot"]
setting_ds_dict = {
    "short-n3":{"base":["c2s"]}, #{"base":["c2s","c2sp","c4s","c4sp"]},
    "medium-n6":{"base":["c2s"]}, #{"base":["c2s","c2sp","c4s","c4sp"]},
    #"long-n9":{"base":["c2s","c2sp","c4s","c4sp"]},
}

for model in models: 
    #df_res = pd.read_csv(f"../data/results/{model}-prompt-results.csv")
    for prompt_setting_key, setting_keys in setting_ds_dict.items():
        for setting_key, ds_values in setting_keys.items():
            for ds in ds_values:
            
                all_refs = []
                all_preds = []
                output_name = f"{ds}-{setting_key}"
                num_examples = prompt_setting_key.split("-")[-1]
                # read-in right df
                df_gen = pd.read_csv(f"../data/gen_predictions/predictions_{model}-{output_name}-{num_examples}.csv")
     
                for index, row in tqdm(df_gen.iterrows(), total=len(df_gen)):
                    ref = row['elaboration_sentence']
                    prediction = row['pred_elaboration'] 
                
                    # Tokenize
                    tokenized_ref = tokenizer(ref).split()
                    tokenized_pred = tokenizer(prediction).split()
                    
                    all_refs.append([tokenized_ref]) 
                    all_preds.append(tokenized_pred)
                
                bleu1_score = corpus_bleu(all_refs, all_preds, weights=(1.0, 0, 0, 0), smoothing_function=smoothing_function)  # 1-gram
                bleu2_score = corpus_bleu(all_refs, all_preds, weights=(0.5, 0.5, 0, 0), smoothing_function=smoothing_function)  # 2-gram
                bleu4_score = corpus_bleu(all_refs, all_preds, weights=(0.25, 0.25, 0.25, 0.25), smoothing_function=smoothing_function)  # 4-gram
    
                """idx = df_res.index[df_res["dataset"] == ds].tolist()[0]
                df_res.at[idx, f"{prompt_setting_key}-b1"] = round(bleu1_score*100,3)
                df_res.at[idx, f"{prompt_setting_key}-b2"] = round(bleu2_score*100,3)
                print(f"{model}-{ds}-{prompt_setting_key} B1: {round(bleu1_score*100,3)}")
                print(f"{model}-{ds}-{prompt_setting_key} B2: {round(bleu2_score*100,3)}")
        
        df_res.to_csv(f"../data/results/{model}-prompt-results.csv",index=False)
        print(f"Results saved for {model}")"""

100%|██████████████████████████████████████| 116/116 [00:00<00:00, 19120.46it/s]


## BERTScore

In [6]:
from tqdm.notebook import tqdm
from bert_score import BERTScorer
from dataset_utils import create_scores_df
import numpy as np
from transformers import logging

# suppress warnings
#logging.set_verbosity_error()

scorer = BERTScorer(model_type='bert-base-uncased',device='cuda:0')

for model in models: 
    #df_res = pd.read_csv(f"../data/results/{model}-prompt-results.csv")
    for prompt_setting_key, setting_keys in setting_ds_dict.items():
        for setting_key, ds_values in setting_keys.items():
            for ds in ds_values:
            
                bert_scores_precision = []
                bert_scores_recall = []
                bert_scores_f1 = []
                
                output_name = f"{ds}-{setting_key}"
                # read-in right df
                num_examples = prompt_setting_key.split("-")[-1]
                # read-in right df
                df_gen = pd.read_csv(f"../data/gen_predictions/predictions_{model}-{output_name}-{num_examples}.csv")
                df_scores = create_scores_df(df_gen)

                #df_scores = create_scores_df(df_gen)
                
                for index, row in tqdm(df_gen.iterrows(), total=len(df_gen)):
                    elaboration = row['elaboration_sentence']
                    prediction = row['pred_elaboration']
                    
                    #  BERTScore for this pair
                    P, R, F1 = scorer.score(
                        cands=[prediction],  
                        refs=[elaboration],              
                    )
                    
                    bert_scores_precision.append(P.mean().item())
                    bert_scores_recall.append(R.mean().item())
                    bert_scores_f1.append(F1.mean().item())
    
                # save result for each pair
                df_scores["bsf1"] = bert_scores_f1
                df_scores.to_csv(f"../data/bert_scores/bert_scores_{model}-{output_name}-{prompt_setting_key}.csv",index=False)
                print(f"Results saved for {model}-{output_name}-{prompt_setting_key}")
            
                
                """avg_f1 = np.mean(bert_scores_f1)
                idx = df_res.index[df_res["dataset"] == ds].tolist()[0]
                df_res.at[idx, f"{prompt_setting_key}-bsf1"] = round(avg_f1,3)
                print(f"{model}-{ds}-{prompt_setting_key}: {round(avg_f1,3)}")
    
        df_res.to_csv(f"../data/results/{model}-prompt-results.csv",index=False)
        print(f"Results saved for {model}")"""

  0%|          | 0/116 [00:00<?, ?it/s]

Results saved for llama-instruct-few-shot-c2s-base-short-n3


  0%|          | 0/116 [00:00<?, ?it/s]

Results saved for llama-instruct-few-shot-c2s-base-medium-n6


## BARTScore

In [8]:
import sys
import os
parent_dir = os.path.abspath('..')
sys.path.append(parent_dir)

from utils.bart_score import BARTScorer
from tqdm.notebook import tqdm
import numpy as np

bart_scorer = BARTScorer(device='cuda:0')

for model in models: 
    df_res = pd.read_csv(f"data/results/{model}-prompt-results.csv")
    for prompt_setting_key, setting_keys in setting_ds_dict.items():
        for setting_key, ds_values in setting_keys.items():
            for ds in ds_values:

                bart_scores = []
                output_name = f"{ds}-{setting_key}"
                # read-in right df
                num_examples = prompt_setting_key.split("-")[-1]
                # read-in right df
                df_gen = pd.read_csv(f"data/gen_predictions/predictions_{model}-{output_name}-{num_examples}.csv")
                df_scores = create_scores_df(df_gen)
            
                for index, row in tqdm(df_gen.iterrows(), total=len(df_gen)):
                    reference = row['elaboration_sentence']  # reference text (r)
                    hypothesis = row['pred_elaboration']    # generated text (h)
                    
                    # precision (r → h)
                    precision_score = bart_scorer.score(
                        srcs=[reference],  # r as source
                        tgts=[hypothesis], # h as target
                        batch_size=1
                    )[0]
                    
                    # recall (h → r)
                    recall_score = bart_scorer.score(
                        srcs=[hypothesis],  # h as source
                        tgts=[reference],   # r as target
                        batch_size=1
                    )[0]
                    
                    # f1 score as the average of precision and recall
                    f1_score = (precision_score + recall_score) / 2
                    bart_scores.append(f1_score)
                
                # save score result for each pair
                df_scores["bartscore"] = bart_scores
                df_scores.to_csv(f"data/bart_scores/bart_scores_{model}-{output_name}-{prompt_setting_key}.csv",index=False)
                print(f"Results saved for {model}-{output_name}-{prompt_setting_key}")
    
                # average score
                """avg_score = np.mean(bart_scores)
                idx = df_res.index[df_res["dataset"] == ds].tolist()[0]
                df_res.at[idx, f"{prompt_setting_key}-bartscore"] = round(avg_score,3)
                print(f"{model}-{ds}-{prompt_setting_key}: {round(avg_score,3)}")
    
                df_res.to_csv(f"data/results/{model}-prompt-results.csv",index=False)
                print(f"Results saved for {model}")"""

  0%|          | 0/116 [00:00<?, ?it/s]

Results saved for llama-instruct-few-shot-c2s-base-short-n3


  0%|          | 0/116 [00:00<?, ?it/s]

Results saved for llama-instruct-few-shot-c2s-base-medium-n6
